# 09 - Full-Sequence Simulation Export (for a FiftyOne-style viewer)

Runs the trained detector over **entire test-sequence videos** - not just the
20 mid-approach frames it was trained on - and exports everything a viewer app
needs, in a self-describing folder per sequence.

Two things worth knowing before reading the output:

- **This covers the whole approach**, including the final metres where the
  object drops below the bumper camera's field of view. The model never saw
  those frames in training (the labeling window is ratio 0.30-0.85), so
  degradation at the very end is *expected* and is exactly what makes this a
  useful simulation.
- **Sequences are drawn from the gold test split** by default - human-clicked
  sequences the model was never trained on - so what you see is honest
  held-out behaviour.

### Export layout

```
simulation/<run_id>/
  index.json                 <- sequence list, class names, model provenance
  <sequence_id>/
    meta.json                <- camera, direction, fps, frame count, target object
    frames/000000.jpg ...    <- every frame of the sequence
    predictions/000000.txt   <- YOLO format + confidence, per frame
    predictions.json         <- COCO-style detections for the whole sequence
    detections.csv           <- flat table: frame, class, category, conf, box
    annotated.mp4            <- boxes burned in
```

`predictions.json` is COCO-shaped (images / annotations / categories) with a
`score` and a deployment `category` on each annotation, so most viewers ingest
it directly; `detections.csv` is there for quick pandas work.

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None
RAW_ROOT_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import csv
import json as _json
from collections import defaultdict

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from tqdm.auto import tqdm
from ultralytics import YOLO

from vision_uss_research.runs import finish_run, new_run_id, resolve_run, start_run
from vision_uss_research.sequences import (driving_direction, sequence_files,
                                           with_drive_retry)

MODEL_RUN = None            # models/<run_id>; None = newest trained run
N_SEQUENCES = 6
CONF = 0.25
SEQUENCE_IDS = []           # explicit sequence ids override the automatic pick
OUT_FPS = 4                 # annotated video fps (source is ~2 fps - slow but jerky)
FRAME_STRIDE = 1            # 1 = every frame
SEED = 0

CLASS_BINS = {"curbstone": "low", "speedbump": "low", "woodenboard": "low",
              "pole": "high", "dummychild": "high", "bicyclestand": "high",
              "low_other": "low", "high_other": "high"}
CATEGORY_NAMES = {"low_other": "Low Unknown", "high_other": "High Unknown"}
def category_of(name):
    return CATEGORY_NAMES.get(name, CLASS_BINS.get(name, "?").capitalize())

model_dir = resolve_run(ARTIFACTS / "models", "train", run_id=MODEL_RUN,
                        require_file="train/weights/best.pt")
assert model_dir is not None, "no trained model - run notebook 06 first"
model_run = _json.loads((model_dir / "run.json").read_text())
dataset_dir = ARTIFACTS / Path(model_run["inputs"][0])
prov = _json.loads((dataset_dir / "provenance.json").read_text())
class_names = prov["class_names"]

weights = model_dir / "train" / "weights" / "best.pt"
model = YOLO(str(weights))

OUT_ROOT = ARTIFACTS / "simulation" / new_run_id("simulation", "viewer")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
start_run(OUT_ROOT, config={"model_run": model_dir.name, "conf": CONF,
                            "n_sequences": N_SEQUENCES, "stride": FRAME_STRIDE},
          inputs=[f"models/{model_dir.name}", f"datasets/{dataset_dir.name}"],
          repo_root=REPO_DIR)
print(f"model    : {model_dir.name}")
print(f"dataset  : {dataset_dir.name}")
print(f"classes  : {class_names}")
print(f"output   : {OUT_ROOT}")

## Pick sequences (gold test split, class-diverse)

In [ ]:
manifest = pd.read_csv(dataset_dir / "dataset_manifest.csv")
test = manifest[manifest["split"] == "test"]
gold = set(prov.get("gold_test_sequences", []))

# one sequence per class where possible, preferring gold (human-clicked) ones
rng = np.random.default_rng(SEED)
id_to_name = {i: n for i, n in enumerate(class_names)}
test = test.assign(class_name=test["class_id"].map(id_to_name))
picked, seen = [], set()
for cname, grp in test.groupby("class_name"):
    ids = sorted(set(grp["sequence_id"].astype(str)))
    pref = [s for s in ids if s in gold] or ids
    choice = pref[int(rng.integers(len(pref)))]
    if choice not in seen:
        picked.append((choice, cname)); seen.add(choice)
# top up / trim to N_SEQUENCES
if len(picked) < N_SEQUENCES:
    extra = [s for s in sorted(set(test["sequence_id"].astype(str))) if s not in seen]
    for s in extra[:N_SEQUENCES - len(picked)]:
        picked.append((s, "")); seen.add(s)
picked = picked[:N_SEQUENCES]

if SEQUENCE_IDS:
    picked = [(s, "") for s in SEQUENCE_IDS]

print(f"{len(picked)} sequences selected:")
for sid, cname in picked:
    print(f"   {sid}   target={cname or '?'}   {'GOLD' if sid in gold else ''}")

## Export

In [ ]:
def export_sequence(seq_id, target_hint=""):
    seq_dir = RAW_ROOT / seq_id
    out = OUT_ROOT / seq_id
    (out / "frames").mkdir(parents=True, exist_ok=True)
    (out / "predictions").mkdir(parents=True, exist_ok=True)

    direction = with_drive_retry(driving_direction, seq_dir)
    cam = {"forward": "front", "backward": "rear"}.get(direction)
    found = with_drive_retry(sequence_files, seq_dir)
    video = found.get(f"{cam}_video") if cam else None
    video = video or found.get("rear_video") or found.get("front_video")
    if video is None:
        return {"sequence_id": seq_id, "error": "no_video"}
    cam = cam or ("rear" if "rear" in video.name.lower() else "front")

    cap = cv2.VideoCapture(str(video))
    fps_src = cap.get(cv2.CAP_PROP_FPS) or 1.0
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    coco_images, coco_anns, rows = [], [], []
    writer, ann_id, kept = None, 1, 0
    idx = -1
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        idx += 1
        if idx % FRAME_STRIDE:
            continue
        h, w = frame.shape[:2]
        stem = f"{kept:06d}"
        cv2.imwrite(str(out / "frames" / f"{stem}.jpg"), frame,
                    [cv2.IMWRITE_JPEG_QUALITY, 92])

        res = model.predict(frame, conf=CONF, verbose=False)[0]
        lines, canvas = [], frame.copy()
        for cls, box, conf in zip(res.boxes.cls.tolist(), res.boxes.xyxy.tolist(),
                                  res.boxes.conf.tolist()):
            name = class_names[int(cls)]
            cat = category_of(name)
            x0, y0, x1, y1 = map(float, box)
            lines.append(f"{int(cls)} {((x0+x1)/2)/w:.6f} {((y0+y1)/2)/h:.6f} "
                         f"{(x1-x0)/w:.6f} {(y1-y0)/h:.6f} {conf:.4f}")
            coco_anns.append({"id": ann_id, "image_id": kept, "category_id": int(cls),
                              "bbox": [round(x0,1), round(y0,1),
                                       round(x1-x0,1), round(y1-y0,1)],
                              "area": round((x1-x0)*(y1-y0),1), "iscrowd": 0,
                              "score": round(float(conf),4), "category": cat})
            ann_id += 1
            rows.append({"frame_idx": kept, "video_frame": idx, "class": name,
                         "category": cat, "conf": round(float(conf),4),
                         "x0": round(x0,1), "y0": round(y0,1),
                         "x1": round(x1,1), "y1": round(y1,1)})
            cv2.rectangle(canvas, (int(x0), int(y0)), (int(x1), int(y1)),
                          (60, 60, 235), 2)
            cv2.putText(canvas, f"{cat}:{name} {conf:.2f}",
                        (int(x0), min(int(y1)+20, h-6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (60, 60, 235), 2)
        (out / "predictions" / f"{stem}.txt").write_text(
            ("\n".join(lines) + "\n") if lines else "")
        coco_images.append({"id": kept, "file_name": f"frames/{stem}.jpg",
                            "width": w, "height": h, "video_frame": idx})

        cv2.rectangle(canvas, (0,0), (w,34), (0,0,0), -1)
        cv2.putText(canvas, f"{seq_id[:30]}  {cam}  frame {kept}/{n_total}",
                    (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0,255,255), 2)
        if writer is None:
            writer = cv2.VideoWriter(str(out / "annotated.mp4"),
                                     cv2.VideoWriter_fourcc(*"mp4v"),
                                     OUT_FPS, (w, h))
        writer.write(canvas)
        kept += 1
    cap.release()
    if writer:
        writer.release()

    (out / "predictions.json").write_text(_json.dumps({
        "info": {"sequence_id": seq_id, "camera": cam,
                 "model_run": model_dir.name, "conf": CONF},
        "images": coco_images, "annotations": coco_anns,
        "categories": [{"id": i, "name": n, "category": category_of(n)}
                       for i, n in enumerate(class_names)]}, indent=1))
    pd.DataFrame(rows).to_csv(out / "detections.csv", index=False)
    meta = {"sequence_id": seq_id, "camera": cam, "driving_direction": direction,
            "target_object_hint": target_hint, "source_video": video.name,
            "source_fps": round(float(fps_src), 3), "frames_exported": kept,
            "frames_in_video": n_total, "stride": FRAME_STRIDE,
            "detections": len(rows),
            "frames_with_detection": int(pd.DataFrame(rows)["frame_idx"].nunique())
                                     if rows else 0,
            "is_gold_test": seq_id in gold}
    (out / "meta.json").write_text(_json.dumps(meta, indent=1))
    return meta

results = []
for sid, cname in tqdm(picked, desc="Simulating sequences", unit="seq"):
    try:
        results.append(export_sequence(sid, cname))
    except Exception as e:
        results.append({"sequence_id": sid, "error": f"{type(e).__name__}: {e}"})
        print(f"FAILED {sid}: {e}")

summary = pd.DataFrame(results)
display(summary)

## Index for the viewer app

In [ ]:
ok = [r for r in results if not r.get("error")]
index = {
    "run_id": OUT_ROOT.name,
    "created_from": {"model_run": model_dir.name, "dataset_run": dataset_dir.name,
                     "weights": str(weights), "conf": CONF},
    "class_names": class_names,
    "categories": {n: category_of(n) for n in class_names},
    "layout": {"frames": "frames/%06d.jpg",
               "labels_yolo_conf": "predictions/%06d.txt",
               "coco": "predictions.json",
               "flat_csv": "detections.csv",
               "video": "annotated.mp4"},
    "sequences": ok,
}
(OUT_ROOT / "index.json").write_text(_json.dumps(index, indent=1))

finish_run(OUT_ROOT, summary={
    "sequences": len(ok),
    "failed": len(results) - len(ok),
    "frames": int(sum(r.get("frames_exported", 0) for r in ok)),
    "detections": int(sum(r.get("detections", 0) for r in ok)),
})
print(f"index      : {OUT_ROOT / 'index.json'}")
print(f"sequences  : {len(ok)}")
print(f"frames     : {sum(r.get('frames_exported', 0) for r in ok)}")
print(f"detections : {sum(r.get('detections', 0) for r in ok)}")
print(f"\nfolder to hand to the viewer app:\n  {OUT_ROOT}")

## Spot-check: first / middle / last annotated frame per sequence

In [ ]:
for r in ok[:6]:
    sid = r["sequence_id"]
    frames = sorted((OUT_ROOT / sid / "frames").glob("*.jpg"))
    if not frames:
        continue
    picks = [frames[0], frames[len(frames)//2], frames[-1]]
    det = pd.read_csv(OUT_ROOT / sid / "detections.csv") if \
        (OUT_ROOT / sid / "detections.csv").stat().st_size > 0 else pd.DataFrame()
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, fp in zip(axes, picks):
        img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
        fi = int(fp.stem)
        if len(det):
            for _, d in det[det["frame_idx"] == fi].iterrows():
                cv2.rectangle(img, (int(d.x0), int(d.y0)), (int(d.x1), int(d.y1)),
                              (235, 60, 60), 2)
                cv2.putText(img, f"{d.category}:{d['class']} {d.conf:.2f}",
                            (int(d.x0), max(int(d.y0)-8, 16)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (235, 60, 60), 2)
        ax.imshow(img); ax.axis("off"); ax.set_title(f"frame {fi}", fontsize=9)
    fig.suptitle(f"{sid}  ({r['camera']}, {r['frames_exported']} frames, "
                 f"{r['detections']} detections)", fontsize=11)
    plt.show()

## Notes for the viewer app

- **`index.json` is the entry point**: it lists every sequence plus the file
  layout, the class -> deployment-category map, and which model produced the
  run, so the app needs no hard-coded paths.
- **Detections carry both labels**: the fine class (`pole`) and the deployment
  category (`High`). Filtering by category gives the Low/High view that matters
  operationally; filtering by class gives the detail.
- **`is_gold_test`** marks sequences the model never trained on - worth
  surfacing in the UI, because those are the honest ones.
- Frames are numbered `000000...` in export order, and each carries its
  original `video_frame` index in `predictions.json` / `detections.csv`, so you
  can trace back to the source video even with `FRAME_STRIDE > 1`.
- Confidence is in the YOLO txt as a 6th column (non-standard but harmless);
  drop it if your parser is strict.